In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
# Installs are quiet (-q) so the output stays readable on a projector.
!pip install -q google-genai

import os, json, time                    # standard library
from google import genai                 # the SDK
from google.genai import types           # config and content types


# ── Your API key ──────────────────────────────────────────────────────
# The key lives in Colab Secrets, never in the notebook. If this cell
# fails, that is almost always why.
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
    if not API_KEY:
        raise ValueError("empty")
except Exception:
    raise SystemExit(
        "\n" + "=" * 68 +
        "\nNo API key found.\n"
        "\n  1. Click the KEY icon in the left sidebar of Colab."
        "\n  2. Click 'Add new secret'."
        "\n  3. Name it exactly:  GEMINI_API_KEY"
        "\n  4. Paste your key from aistudio.google.com"
        "\n  5. Turn ON 'Notebook access' for this notebook."
        "\n  6. Run this cell again."
        "\n" + "=" * 68
    )

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-2.5-flash-lite"          # fast and cheap; what we use all week
EMBED_MODEL = "gemini-embedding-001"    # free tier, which is what makes Day 2 possible

print("Ready. Model:", MODEL)

# Day 1 — From your first call to typed output

**Applied Generative AI · SDAIA Academy · Musa Ibn Rashid**

### By the end of this notebook you will have

A function called `ask()` that sends a prompt and returns **JSON matching a
schema you defined**, which your own code can branch on — with a system
prompt, a temperature setting and retries built in.

That last part is the whole point of today. A model that returns free text is
a chat toy. A model that returns validated JSON is a **component in a system**.

We get there in four steps: make a call, read the token counts, control the
output, then constrain it with a schema.

In [ ]:
# Your first call. Four lines. Run it — nothing to write here.
resp = client.models.generate_content(
    model=MODEL,
    contents="Explain what a token is, in one sentence, for a beginner.",
)

print(resp.text)

### What just happened

1. Your text was **tokenised** — cut into pieces the model has seen before.
2. Those tokens went through the model, which predicted the **next token**.
3. It appended that token and repeated, hundreds of times.
4. The result was decoded back into text.

Nothing in that loop checked whether the answer was *true*. It checked whether
it was *likely*. Hold on to that — it explains almost every failure this week.

In [ ]:
# Every response carries its token counts. This is your bill, itemised.
u = resp.usage_metadata

print("prompt tokens :", u.prompt_token_count)
print("output tokens :", u.candidates_token_count)
print("total tokens  :", u.total_token_count)

# Note: attribute names come from the SDK's usage metadata object. If your
# installed version differs, run  dir(resp.usage_metadata)  to see them.

In [ ]:
# TODO ─ Send a prompt of your own and note the token count.
#
# Replace the text on the line marked TODO. Try something from your actual
# work — a question about a policy, a summary request, anything.

my_prompt = "TODO: write your own prompt here"        # ← TODO (1 line)

resp2 = client.models.generate_content(model=MODEL, contents=my_prompt)

print(resp2.text)
print()
print("in:", resp2.usage_metadata.prompt_token_count,
      "| out:", resp2.usage_metadata.candidates_token_count)

In [ ]:
# The same sentence, in English and in Arabic. Pre-written — just run it.
english = "Personal data may not be shared with any external party without written consent."
arabic  = "لا يجوز مشاركة البيانات الشخصية مع أي جهة خارجية دون موافقة مكتوبة."

# count_tokens asks the API to tokenise without generating anything.
en = client.models.count_tokens(model=MODEL, contents=english)
ar = client.models.count_tokens(model=MODEL, contents=arabic)

print("English :", en.total_tokens, "tokens |", len(english), "characters")
print("Arabic  :", ar.total_tokens, "tokens |", len(arabic), "characters")
print()
print("Arabic costs %.1fx the tokens for the same meaning." % (ar.total_tokens / en.total_tokens))

### Why Arabic costs more, and why that is a budget line

Tokenizers are trained mostly on English text, so English words are usually
one token each. Arabic words break into several smaller pieces.

Same meaning, two to three times the tokens — **on the input and on the
output, on every single call, for the life of the product**.

On Wednesday we put real prices against this. An assistant that costs $18 a
month in English costs closer to $45 in Arabic, for identical usage. Nobody
warns you about this in advance. Now you know.

In [ ]:
# Temperature 0.0 — the model always takes the most likely next token.
# Run the same prompt three times and compare.
prompt = "Describe a data governance policy in one short sentence."

for i in range(3):
    r = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.0),
    )
    print(i + 1, "|", r.text.strip())

In [ ]:
# Temperature 1.2 — the same prompt, much flatter sampling.
for i in range(3):
    r = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=1.2),
    )
    print(i + 1, "|", r.text.strip())

### TODO — reflection

Look at the two cells above.

**In one sentence, describe the difference you saw:**

> _your answer here_

**Which temperature would you use for extracting a person's name and email
from a support message, and why?**

> _your answer here_

In [ ]:
# A system instruction sets who the model is. Same user prompt, two personas.
question = "An employee asks whether they can carry over 15 days of leave."

for persona in [
    "You are a formal HR policy officer. Answer in one precise sentence.",
    "You are a friendly colleague explaining over coffee. Two casual sentences.",
]:
    r = client.models.generate_content(
        model=MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=persona,      # ← the only thing changing
            temperature=0.3,
        ),
    )
    print("PERSONA:", persona)
    print(r.text.strip())
    print("-" * 60)

In [ ]:
# TODO ─ Write a system instruction that forces the answer into
#        bullet points, and under 50 words.
#
# The structure is written. Fill in the one line marked TODO. Be explicit:
# vague instructions produce vague compliance.

my_system = "TODO: your system instruction here"      # ← TODO (1 line)

r = client.models.generate_content(
    model=MODEL,
    contents="Summarise why retrieval-augmented generation is useful.",
    config=types.GenerateContentConfig(
        system_instruction=my_system,
        temperature=0.2,
    ),
)

print(r.text)
print()
print("Word count:", len(r.text.split()), "(target: under 50)")

In [ ]:
# The context window is finite. Here we deliberately overflow it and catch
# the error, so you recognise it when it happens for real.
huge = "The quick brown fox jumps over the lazy dog. " * 400_000   # ~3.6M words

try:
    r = client.models.generate_content(model=MODEL, contents=huge)
    print(r.text[:200])
except Exception as e:
    print("Failed, as expected.")
    print(type(e).__name__, ":", str(e)[:300])

# This is why chunking exists tomorrow. You cannot paste the manual in.

## The ceiling starts here — why free text is unusable in software

Ask for a name and an urgency level three times and you get three shapes:

```
"The customer is Sara Al-Otaibi and this seems quite urgent."
"Name: Sara Al-Otaibi\nUrgency: High"
"Sure! Here's what I found — the sender appears to be Sara…"
```

Now write the `if` statement that routes the urgent ones. You cannot, not
reliably. The instinct is to write a smarter parser; that is treating a
symptom.

The fix is a **contract**: tell the model the exact shape of the response and
have the API enforce it. That is the next three cells, and it is the most
under-taught idea in this field.

In [ ]:
# JSON mode with a response schema. Pre-written — read it carefully.
message = """
From: sara.alotaibi@example.gov.sa
Subject: URGENT - portal down before deadline

The training request portal has been down since 6am. I have a submission
deadline at noon today and cannot file form SDAIA-F-CRS-201-01-V1.
"""

# Ordinary JSON Schema — the same thing you would write for any API.
schema = {
    "type": "object",
    "properties": {
        "name":    {"type": "string"},
        "email":   {"type": "string"},
        # enum means the model CANNOT return "quite urgent".
        "intent":  {"type": "string", "enum": ["question", "complaint", "request", "outage"]},
        "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": ["name", "email", "intent", "urgency"],
}

resp = client.models.generate_content(
    model=MODEL,
    contents="Extract the sender details from this message:\n" + message,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",   # no prose, no markdown fence
        response_schema=schema,                  # the contract itself
        temperature=0.0,                         # extraction: no creativity
    ),
)

print(resp.text)

In [ ]:
# THIS is the moment. Parse it, then branch on it in ordinary Python.
data = json.loads(resp.text)          # a real dict, with keys you chose

print("Parsed:", data)
print()

if data["urgency"] == "high":
    print("→ page the duty officer for", data["name"])
elif data["intent"] == "complaint":
    print("→ open a ticket in the complaints queue")
else:
    print("→ reply from the standard template")

# No regex. No string matching. No "if the answer contains the word urgent".
# The rest of your system does not need to know an LLM was involved.

In [ ]:
# TODO ─ Define your own schema for a different extraction task.
#
# Pick something from your own work: extracting fields from a request form,
# classifying an incident, pulling structured data out of a report.
#
# Two lines are blanked. The rest is written.

my_text = """
Ticket 4471: The meeting room booking system rejected a board room request
from the finance department yesterday afternoon. Reported by Ahmed Al-Qahtani.
Not urgent, but it has happened three times this month.
"""

my_schema = {
    "type": "object",
    "properties": {
        # ← TODO (2 lines): define at least three fields you want back.
        # At least one of them must use "enum" to constrain the values.
        # Example shape:  "department": {"type": "string"},
    },
    "required": [],       # ← list the field names you defined above
}

r = client.models.generate_content(
    model=MODEL,
    contents="Extract the structured fields from:\n" + my_text,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=my_schema,
        temperature=0.0,
    ),
)

print(r.text)
print()
print("As a dict:", json.loads(r.text))

In [ ]:
# The ask() wrapper. GIVEN COMPLETE — read it, do not retype it.
# You will import this idea every day for the rest of the week.

def ask(prompt, system=None, temperature=0.2, schema=None, tries=3):
    """One call, with a system prompt, retries, and optional JSON schema.

    Returns a parsed dict when schema is given, otherwise plain text.
    """
    # Build the config. Only ask for JSON when a schema was supplied.
    opts = {"temperature": temperature}
    if system:
        opts["system_instruction"] = system
    if schema:
        opts["response_mime_type"] = "application/json"
        opts["response_schema"] = schema

    cfg = types.GenerateContentConfig(**opts)

    for attempt in range(tries):
        try:
            r = client.models.generate_content(
                model=MODEL, contents=prompt, config=cfg)
            return json.loads(r.text) if schema else r.text

        except Exception as e:
            # Last attempt? Give up honestly rather than returning nonsense.
            if attempt == tries - 1:
                raise
            wait = 2 ** attempt          # 1s, 2s, 4s — exponential backoff
            print(f"  attempt {attempt + 1} failed ({type(e).__name__}), "
                  f"retrying in {wait}s")
            time.sleep(wait)


# Try it both ways.
print(ask("Name three benefits of retrieval-augmented generation.", temperature=0.4))
print()
print(ask("Extract the fields.\n" + message, schema=schema))

## Reflection

Fill these in before you close the notebook. This is what I check when I come round.

**Which cell surprised you most, and why?**

> _your answer here_

**In one sentence: what does a response schema give you that a well-written prompt does not?**

> _your answer here_

**Name one thing at your work that could use the ask() function with a schema. What fields would the schema have?**

> _your answer here_

## If this breaks

The three most likely failures, and what to do about each.

| Symptom | Cause | Fix |
|---|---|---|
| `SystemExit` on cell 1 | The key is not in Colab Secrets, or notebook access is off | Key icon in the left sidebar → add `GEMINI_API_KEY` → toggle notebook access ON → re-run |
| `404` or `model not found` | Model name typo, or that model is not on your key | Check `MODEL` is exactly `gemini-2.5-flash-lite` |
| `json.JSONDecodeError` | You parsed a response that had no schema, so it came back as prose | Pass both `response_mime_type` and `response_schema`, as in cell 16 |